# EL-GHALI MOHAMED

### NAIVE BAYES
Le classifieur Naive Bayes (Bayes Naïf) repose sur le Théorème de Bayes. Il est dit "naïf" car il fait l'hypothèse très forte que toutes les caractéristiques sont indépendantes les unes des autres

Formule de base :


$$P(Classe | Données) \propto P(Classe) \times \prod_{i=1}^{n} P(Caractéristique_i | Classe)$$

La phase d'entraînement consiste simplement à calculer et stocker deux types de probabilités :

- Probabilités a priori (Priors) : La probabilité globale de chaque classe (ex: Quelle est la probabilité de jouer au football en général ?).

- Vraisemblances (Likelihoods) : La probabilité d'observer une certaine caractéristique sachant la classe (ex: Sachant qu'on a joué au football, quelle était la probabilité qu'il fasse Ensoleillé ?).


In [1]:
import pandas as pd

# Le dataset Jouer football classique
data = {
    'Perspectives': ['Ensoleille', 'Ensoleille', 'Couvert', 'Pluie', 'Pluie', 'Pluie', 'Couvert',
                     'Ensoleille', 'Ensoleille', 'Pluie', 'Ensoleille', 'Couvert', 'Couvert', 'Pluie'],
    'Temperature': ['Chaude', 'Chaude', 'Chaude', 'Douce', 'Fraiche', 'Fraiche', 'Fraiche',
                    'Douce', 'Fraiche', 'Douce', 'Douce', 'Douce', 'Chaude', 'Douce'],
    'Humidite': ['Haute', 'Haute', 'Haute', 'Haute', 'Normale', 'Normale', 'Normale',
                 'Haute', 'Normale', 'Normale', 'Normale', 'Haute', 'Normale', 'Haute'],
    'Vent': ['Faible', 'Fort', 'Faible', 'Faible', 'Faible', 'Fort', 'Fort',
             'Faible', 'Faible', 'Faible', 'Fort', 'Fort', 'Faible', 'Fort'],
    'Jouer': ['Non', 'Non', 'Oui', 'Oui', 'Oui', 'Non', 'Oui',
              'Non', 'Oui', 'Oui', 'Oui', 'Oui', 'Oui', 'Non']
}
df = pd.DataFrame(data)
print(df.head())

  Perspectives Temperature Humidite    Vent Jouer
0   Ensoleille      Chaude    Haute  Faible   Non
1   Ensoleille      Chaude    Haute    Fort   Non
2      Couvert      Chaude    Haute  Faible   Oui
3        Pluie       Douce    Haute  Faible   Oui
4        Pluie     Fraiche  Normale  Faible   Oui


### Entraînement

In [2]:
def entrainer_naive_bayes(donnees, colonne_cible):
    """Calcule les probabilités a priori et les vraisemblances."""

    # 1. Calcul des probabilités a priori : P(Classe)
    # Ex: P(Jouer=Oui) = 9/14, P(Jouer=Non) = 5/14
    classes = donnees[colonne_cible].unique()
    total_lignes = len(donnees)

    priors = {}
    for c in classes:
        priors[c] = len(donnees[donnees[colonne_cible] == c]) / total_lignes

    #  Calcul des vraisemblances : P(Feature | Classe)
    # Ex: P(Perspectives=Ensoleille | Jouer=Oui)
    likelihoods = {}
    colonnes_features = donnees.columns.drop(colonne_cible)

    for feature in colonnes_features:
        likelihoods[feature] = {}
        valeurs_possibles = donnees[feature].unique()

        for valeur in valeurs_possibles:
            likelihoods[feature][valeur] = {}

            for c in classes:
                # On isole les lignes correspondant à la classe actuelle (Ex: Que les jours 'Oui')
                sous_ensemble_classe = donnees[donnees[colonne_cible] == c]
                total_classe = len(sous_ensemble_classe)

                # Combien de fois cette valeur apparaît dans cette classe ?
                occurences_valeur = len(sous_ensemble_classe[sous_ensemble_classe[feature] == valeur])

                # Calcul de la probabilité conditionnelle
                probabilite = occurences_valeur / total_classe

                likelihoods[feature][valeur][c] = probabilite

    print("Modèle Naive Bayes entraîné (Probabilités calculées) !")
    return priors, likelihoods

### Prédiction

In [3]:
def predire_naive_bayes(requete, priors, likelihoods):
    """Calcule la probabilité d'appartenance à chaque classe pour une nouvelle donnée."""
    scores = {}

    for c in priors:
        # On initialise le score avec la probabilité de base de la classe
        score_classe = priors[c]

        # On multiplie par les probabilités de chaque caractéristique de notre requête
        for feature, valeur in requete.items():
            # Si la valeur existe dans notre modèle
            if valeur in likelihoods[feature]:
                score_classe *= likelihoods[feature][valeur][c]
            else:
                # Si la valeur n'a jamais été vue, on met une probabilité très faible
                score_classe *= 0.0001

        scores[c] = score_classe

    #  Convertir les scores bruts en pourcentages lisibles
    total_scores = sum(scores.values())
    probabilites = {}
    for c in scores:
        probabilites[c] = (scores[c] / total_scores) * 100

    # Trouver la classe avec la plus haute probabilité
    meilleure_classe = max(probabilites, key=probabilites.get)

    return meilleure_classe, probabilites

### Lancement et Test

In [4]:
cible = 'Jouer'

# Entraînement
mes_priors, mes_likelihoods = entrainer_naive_bayes(df, cible)

# Test avec une nouvelle journée
journee_test = {
    'Perspectives': 'Pluie',
    'Temperature': 'Fraiche',
    'Humidite': 'Normale',
    'Vent': 'Fort'
}

# Prédiction
prediction, probas = predire_naive_bayes(journee_test, mes_priors, mes_likelihoods)

print("\n=== RÉSULTATS DE LA PRÉDICTION ===")
print(f"Météo observée : {journee_test}")
print("\nProbabilités calculées :")
for classe, proba in probas.items():
    print(f" - Probabilité de '{classe}' : {proba:.2f}%")

print(f"\n=> DÉCISION DU MODÈLE : {prediction}")

Modèle Naive Bayes entraîné (Probabilités calculées) !

=== RÉSULTATS DE LA PRÉDICTION ===
Météo observée : {'Perspectives': 'Pluie', 'Temperature': 'Fraiche', 'Humidite': 'Normale', 'Vent': 'Fort'}

Probabilités calculées :
 - Probabilité de 'Non' : 17.76%
 - Probabilité de 'Oui' : 82.24%

=> DÉCISION DU MODÈLE : Oui
